In [1]:
setwd('/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/')
devtools::load_all('utils/modules/R/prstools')

i Loading PRStools

Loading required package: bigsnpr

Loading required package: bigstatsr

Loading required package: data.table

Loading required package: bigassertr

! Adding files missing in collate: /gpfs3/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/utils/modules/R/prstools/R/qc_sumstat.R



In [2]:
args <- list(
  chrom = "chr21",
  pred = "data/prs/hapmap/ukb_500k/ukb_hapmap_500k_eur_chr21.bed",
  gwas = "data/prs/sumstat/cts/by_chrom/ukb_hapmap_500k_eur_Alanine_aminotransferase_chr21.txt.gz",
  ldsc = "data/prs/ldsc/ldsc_CAD_combined.rds",
  ld_dir = "data/prs/hapmap/ld/matrix",
  ld_bed = "data/prs/hapmap/ld/unrel_eur_10k/short_merged_ukb_hapmap_rand_10k_eur.bed",
  method = "inf"
)

lapply(args, file.exists)

$chrom
[1] FALSE

$pred
[1] TRUE

$gwas
[1] TRUE

$ldsc
[1] TRUE

$ld_dir
[1] TRUE

$ld_bed
[1] TRUE

$method
[1] FALSE

In [3]:
gwas <- read_hail_sumstat(args$gwas, trait = "cts")

In [4]:
pred <- load_bigsnp_from_bed(args$ld_bed)

In [5]:
info_snp <- snp_match(gwas, pred$map, join_by_pos = TRUE, strand_flip = FALSE)

16,726 variants to be matched.

Some duplicates were removed.

16,686 variants have been matched; 0 were flipped and 0 were reversed.



In [6]:
head(gwas)

chr,pos,rsid,a1,a0,n_eff,beta_se,p,beta,INFO,MAF
<chr>,<int>,<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>
chr21,10510446,chr21:10510446_A_G,G,A,363339,0.0023193,0.19934,0.00297660,NA,0.051302
chr21,13222943,chr21:13222943_T_A,A,T,363339,0.0037778,0.89046,0.00052031,NA,0.018222
chr21,13258267,chr21:13258267_A_G,G,A,363339,0.0010081,0.67615,-0.00042109,NA,0.472800
chr21,13268079,chr21:13268079_G_A,A,G,363339,0.0010081,0.70384,-0.00038321,NA,0.472810
chr21,13270143,chr21:13270143_A_G,G,A,363339,0.0010084,0.67495,-0.00042290,NA,0.472670
chr21,13277477,chr21:13277477_C_A,A,C,363339,0.0010088,0.47966,-0.00071309,NA,0.477230


In [8]:
res <- qc_sumstat(pred$G, info_snp, trait = "cts")

In [9]:
sum(res$is_bad)

[1] 546

In [36]:
G <- pred$G

In [71]:
maf <- snp_MAF(G, ind.col = info_snp$`_NUM_ID_`, ncores = 1)

In [42]:
n <- unique(gwas$n)

In [57]:
min(sqrt(0.5) * info_snp$beta_se^2 * n)

[1] 0.2524176

In [64]:
sd_val <- sqrt(2 * maf * (1 - maf))
sd_y <- min(sqrt(0.5) * info_snp$beta_se^2 * n)
sd_ss <- with(info_snp, sd_y / sqrt(n_eff * beta_se^2 + beta^2))

In [70]:
min(sqrt(0.5) * info_snp$beta_se^2 * n)

[1] 0.2524176

In [65]:
is_bad_sd <-
      sd_ss < (0.5 * sd_val) | sd_ss > (sd_val + 0.1) | sd_ss < 0.1 | sd_val < 0.05

In [66]:
is_bad_maf <- is.na(maf)
is_bad <- (is_bad_sd | is_bad_maf)
sum_is_bad <- sum(is_bad, na.rm = TRUE)
sum_n <- length(is_bad)
print(paste(sum_is_bad, 'of', sum_n, 'SNPs fail QC'))


[1] "546 of 16686 SNPs fail QC"


In [49]:
qc_binary_sumstat <- function(G, info_snp, n_eff, ncores = 1){
    maf <- snp_MAF(G, ind.col = info_snp$`_NUM_ID_`, ncores = ncores)
    sd_val <- sqrt(2 * maf * (1 - maf))
    sd_ss <- with(info_snp, 2 / sqrt(n_eff * beta_se^2 + beta^2))
    is_bad_sd <-
      sd_ss < (0.5 * sd_val) | sd_ss > (sd_val + 0.1) | sd_ss < 0.1 | sd_val < 0.05
    is_bad_maf <- is.na(maf)
    is_bad <- (is_bad_sd | is_bad_maf)
    sum_is_bad <- sum(is_bad, na.rm = TRUE)
    sum_n <- length(is_bad)
    write(paste(sum_is_bad, 'of', sum_n, 'SNPs fail QC'), stderr())
    return(list(is_bad = is_bad, sd_val = sd_val, sd_ss = sd_ss))
}

In [3]:
# laod ldsc and qced GWAS
  ldsc <- readRDS(args$ldsc)
  gwas <- ldsc$gwas
  h2 <- ldsc$h2_est

  # Estimate h2 chromosome-wide
  N_total <- nrow(gwas)
  N_chr <- sum(gwas$chr == args$chr)
  h2_init <- h2 * (N_chr / N_total)

  # check estimates
  stopifnot(h2_init > 0)
  stopifnot(!is.null(gwas))



In [4]:
  # get SNP correlations and LD
  snp <- get_single_ld_matrix(gwas, chr = args$chrom, ld_dir = args$ld_dir)

  # match GWAS with snp-correlation map
  indicies <- na.omit(match(snp$map$marker, gwas$marker))
  gwas <- gwas[indicies,]

  # check that LD-matrix markers and gwas markers have overlap
  # Check that ordering of markers are actually matching
  stopifnot(all(gwas$marker %in% snp$map$marker))
  stopifnot(all(snp$map$marker %in% gwas$marker))
  stopifnot(sum(gwas$marker == snp$map$marker) / nrow(gwas) == 1)


In [5]:
   # load data to be used for prediction
  pred <- load_bigsnp_from_bed(args$pred)
  pred <- match_bigsnp_with_gwas(pred, gwas)
  genotypes <- pred$genotypes
  indicies <- pred$gwas_indciess

In [6]:
dim(snp$corr); nrow(gwas); beta_inf

[1] 16307 16307

[1] 16307

ERROR: Error in eval(expr, envir, enclos): object 'beta_inf' not found


[1] 16307

In [ ]:
beta_inf <- snp_ldpred2_inf(snp$corr, gwas, h2_init)
head(beta_inf); length(beta_inf)

In [8]:
  write("Running multi_auto", stderr())
     multi_auto <- snp_ldpred2_auto(
        corr = snp$corr,
        df_beta = gwas,
        h2_init = h2_init,
        vec_p_init = seq_log(1e-4, 0.5, 30),
        verbose = TRUE,
        ncores = 1)

   

1: 0.000126895 // 0.000291787
2: 0.000128249 // 0.000635245
3: 0.000260636 // 0.000441418
4: 0.000325939 // 0.00049687
5: 0.000792083 // 0.00151865
6: 0.000603019 // 0.00121833
7: 0.000564929 // 0.000882717
8: 0.00103343 // 0.000913659
9: 0.000901696 // 0.00136474
10: 0.00155807 // 0.00128785
11: 0.000984423 // 0.000844179
12: 0.00193337 // 0.00128503
13: 0.00293289 // 0.00175614
14: 0.00349212 // 0.0015123
15: 0.00442031 // 0.00141285
16: 0.00450728 // 0.00145861
17: 0.00516121 // 0.00139733
18: 0.00453361 // 0.00141943
19: 0.00438279 // 0.0014346
20: 0.00418073 // 0.0016225
21: 0.00438242 // 0.00193025
22: 0.0048254 // 0.00191395
23: 0.00500898 // 0.00211204
24: 0.00617531 // 0.00223179
25: 0.00500974 // 0.00193889
26: 0.00444356 // 0.00218321
27: 0.00475744 // 0.00225841
28: 0.00540014 // 0.00213315
29: 0.00529293 // 0.00242545
30: 0.00486154 // 0.00212868
31: 0.00588552 // 0.0017127
32: 0.00615944 // 0.00159509
33: 0.00657214 // 0.00207463
34: 0.00592249 // 0.0017774
35: 0.00713606

In [39]:
saveRDS(multi_auto, 'test.rda', compress = 'xz')

In [12]:
str(multi_auto)

List of 30
 $ :List of 9
  ..$ beta_est   : num [1:16307] -2.38e-04 -6.17e-06 -5.82e-06 -7.43e-06 2.78e-06 ...
  ..$ postp_est  : num [1:16307] 0.01017 0.00672 0.0067 0.00675 0.00666 ...
  ..$ sample_beta: num[1:16307, 0 ] 
  ..$ p_est      : num 0.0103
  ..$ h2_est     : num 0.00228
  ..$ path_p_est : num [1:1500] 0.000127 0.000128 0.000261 0.000326 0.000792 ...
  ..$ path_h2_est: num [1:1500] 0.000292 0.000635 0.000441 0.000497 0.001519 ...
  ..$ h2_init    : num 0.00196
  ..$ p_init     : num 1e-04
 $ :List of 9
  ..$ beta_est   : num [1:16307] -1.84e-04 -4.94e-06 -4.71e-06 -5.74e-06 1.56e-06 ...
  ..$ postp_est  : num [1:16307] 0.00705 0.00441 0.00441 0.00443 0.00436 ...
  ..$ sample_beta: num[1:16307, 0 ] 
  ..$ p_est      : num 0.00744
  ..$ h2_est     : num 0.00215
  ..$ path_p_est : num [1:1500] 9.42e-05 2.21e-04 4.93e-04 1.87e-03 2.97e-03 ...
  ..$ path_h2_est: num [1:1500] 0.00048 0.000555 0.000502 0.001309 0.001438 ...
  ..$ h2_init    : num 0.00196
  ..$ p_init     : num 0.

In [14]:
  # get estimates with indicies corresponding to pred genotypes
     write("Get estimates for beta_auto..", stderr())
     beta_auto <- sapply(multi_auto, function(auto){
          auto$beta_est})

   

In [16]:
str(beta_auto)
head(beta_auto[[1]])
length(beta_auto)

 num [1:16307, 1:30] -2.38e-04 -6.17e-06 -5.82e-06 -7.43e-06 2.78e-06 ...


[1] -0.0002376847

[1] 489210

In [18]:
dim(pred$genotypes)

[1] 381244  16307

In [20]:
  # perform matrix multiplication
     write("Performing mat mul..", stderr())
     pred_auto <- big_prodMat(
        genotypes,
        beta_auto,
        ncores = 1)


In [41]:

     # quality controls on chains
     write("Quality control om chains..", stderr())
     sc <- apply(pred_auto, 2, sd, na.rm = TRUE)
     keep <- abs(sc - median(sc)) < 3 * mad(sc)
     final_beta_auto <- rowMeans(beta_auto[, keep], na.rm = TRUE)

  

In [42]:
final_beta_auto

[1] -2.177340e-04 -5.704885e-06 -5.488641e-06 -6.720491e-06  2.374155e-06
    [6] -1.178676e-05 -7.723547e-06 -4.838455e-05 -1.463803e-06  1.054120e-05
   [11] -6.031848e-07  1.871242e-04  1.124993e-05 -2.370763e-05  2.173058e-04
   [16] -3.876444e-06  1.652801e-05  1.113446e-05  9.554647e-06 -1.792740e-05
   [21]  3.177385e-05 -4.274642e-07  9.293229e-06  2.171322e-05 -2.852639e-05
   [26] -6.629152e-05 -1.155975e-04 -1.232165e-05  7.900129e-06 -1.903423e-05
   [31] -1.616393e-05 -4.179153e-04 -1.729425e-04 -1.723422e-04 -1.347899e-04
   [36] -4.694311e-04 -3.024152e-04 -1.654447e-04  1.374985e-05  2.873700e-07
   [41] -1.625602e-04  7.846467e-06 -4.102815e-06  6.011973e-06 -4.551583e-06
   [46]  1.313914e-03  4.729610e-06 -2.334440e-05  3.132471e-05  1.447960e-05
   [51]  1.236894e-04  1.901781e-05  2.041550e-05  1.257186e-04  6.522036e-06
   [56] -1.131780e-08  1.916309e-05  2.464375e-06  1.956434e-05  3.643991e-05
   [61]  3.211215e-05  2.794592e-05  2.054104e-05  8.413646e-06 -7.358024e-05
   [66] -7.512988e-06  2.754370e-05 -2.486205e-05  2.478843e-05  2.418480e-05
   [71] -1.451457e-05 -4.426968e-05 -1.234580e-07  1.322138e-05 -1.256651e-05
   [76] -1.318049e-05 -3.965148e-06 -4.968767e-06  1.294403e-06  9.088925e-07
   [81]  3.555432e-07 -3.178638e-06  3.408002e-05  7.471977e-06 -1.424410e-05
   [86] -1.203817e-05 -4.373923e-06  2.365097e-06 -9.483985e-06 -8.909134e-06
   [91] -8.927733e-07 -1.596290e-04  7.816466e-06 -1.848507e-05 -1.994619e-06
   [96]  1.485276e-05  1.749486e-05 -8.976321e-06  4.467570e-06 -1.044276e-05
  [101]  4.366044e-06 -1.590738e-05  1.928600e-05  1.500374e-05 -2.431770e-06
  [106] -1.091586e-04  1.144611e-05  1.164911e-05  2.039023e-05 -3.994265e-06
  [111]  9.698839e-06  4.347327e-06 -1.090225e-05 -2.457288e-06  9.771057e-06
  [116]  1.149270e-05  8.177963e-06 -2.028047e-05  6.502342e-06 -1.902980e-05
  [121]  8.941444e-06  1.950608e-05 -1.347915e-04 -1.370862e-04 -6.183925e-05
  [126] -1.367967e-04 -3.027108e-04  7.679912e-06 -1.248951e-04 -1.346545e-04
  [131] -9.826914e-05 -1.347087e-04  2.081352e-05 -1.309211e-04 -1.306774e-04
  [136] -1.342735e-04 -5.909826e-04  5.430597e-05  4.723992e-05 -8.462845e-05
  [141] -8.513788e-05 -8.566416e-05 -8.405943e-05 -2.335485e-05 -2.879629e-05
  [146] -2.297816e-05 -3.979976e-05 -1.867609e-05  6.215829e-05 -3.689134e-05
  [151]  5.341776e-05  5.581421e-05  1.527740e-04  3.484293e-05  3.633147e-05
  [156] -4.636263e-05 -8.788635e-06 -5.481881e-05 -4.946786e-05 -2.664207e-05
  [161] -6.647455e-05  2.096884e-06  1.266882e-05 -1.259620e-05 -3.893723e-05
  [166]  1.420111e-05  1.645984e-05  1.350051e-05 -9.032847e-05 -3.268762e-05
  [171] -8.775141e-05  2.178173e-04 -6.365180e-05 -7.157044e-05  3.019625e-05
  [176] -3.490384e-06 -2.464888e-05 -6.809527e-07 -1.478913e-06 -1.311387e-05
  [181]  2.418171e-05 -4.112013e-05 -3.565101e-06  3.247846e-05 -4.777516e-05
  [186] -4.835307e-06 -1.107387e-05 -1.134690e-04 -2.162730e-05  4.007599e-05
  [191] -4.178280e-05 -2.232883e-05  1.002280e-04  2.538211e-05 -6.121206e-05
  [196] -5.207398e-06  2.022384e-05  3.967682e-05 -1.726501e-05  3.962995e-05
  [201] -8.017751e-06  2.078015e-05  2.017477e-05  3.653509e-05  5.985301e-05
  [206] -3.626045e-06  1.890525e-04  2.224973e-05 -2.204961e-04  3.010770e-04
  [211]  2.765388e-05  2.044891e-04  4.268432e-04  3.941604e-04  2.237509e-04
  [216]  7.608681e-05  4.108654e-05  4.956209e-05  2.238674e-04  9.687159e-05
  [221]  8.025744e-05  8.567141e-05  3.867175e-05  8.673818e-05  9.000530e-05
  [226]  3.818780e-05 -1.414702e-05  2.105465e-05  5.508601e-05  6.581290e-05
  [231]  4.254578e-05  1.888244e-05 -1.764350e-05  6.833752e-05  5.984034e-05
  [236]  7.033949e-05  1.042632e-05 -5.681362e-05  4.473245e-05  2.850871e-05
  [241]  2.623627e-05 -1.325524e-05 -4.441713e-06 -1.817841e-05  3.675257e-06
  [246] -6.204143e-09 -4.230337e-06  4.191701e-06  5.846866e-07 -3.316684e-06
  [251]  3.013563e-05 -2.336716e-04  1.665802e-05  2.019117e-06  1.344766e-05
  [256] -1.237294e-04  1.4

In [37]:
any(is.na(pred_auto))
sum(is.na(pred_auto))
NA_rows <- rowSums(is.na(pred_auto)) > 0 

sum(NA_rows)/nrow(pred_auto)

[1] TRUE

[1] 291150

[1] 0.02545614

In [43]:
   # get final predicton
     write("Getting final prediction..", stderr())
     final_pred_auto <- big_prodVec(
       genotypes,
       final_beta_auto,
       ncores = 1)

     final_pred <- final_pred_auto

In [3]:
# load data required for setting up LD-matrix
ld_data <- load_bigsnp_from_bed(args$bed)

# load summary statistics
sumstats <- read_hail_sumstat(args$gwas)

# match summary stats and LD data
info_snp <- snp_match(sumstats, ld_data$map, join_by_pos = TRUE, strand_flip = FALSE)

ERROR: Error in trait %in% c("binary", "cts"): argument "trait" is missing, with no default


In [ ]:
#QC summary statistics based on LD reference
qc <- qc_binary_sumstat(ld_data$G, info_snp, NCORES)
well_behaved_snps <- (!qc$is_bad)
df_beta <- info_snp[well_behaved_snps, ]
df_beta$marker <- get_ldpred_marker(df_beta)

In [ ]:
snp1 <- get_ld_matrix(df_beta, chrs = 1:22)

In [6]:
snp <- snp1
gwas <- df_beta #df_beta[df_beta$chr %in% paste0("chr",1:2),]
gwas <- gwas[gwas$marker %in% snp$map$marker,]

#gwas <- gwas[order(as.numeric(gsub('chr','',gwas$chr))),] # works!
indicies <- na.omit(match(snp$map$marker, gwas$marker))
gwas <- gwas[indicies,]


nrow(gwas); length(snp$map$marker); length(snp$ld)



# check that LD and GWAS data was generated simultanously
stopifnot(length(snp$ld) == nrow(gwas))
stopifnot(sum(gwas$marker == snp$map$marker) / nrow(gwas) == 1)

[1] 1121294

[1] 1121294

[1] 1121294

In [7]:
# perform LD regressio

chi2 <- (gwas$beta / gwas$beta_se)^2
ldsc <- with(gwas,
            snp_ldsc(
                snp$ld,
                length(snp$ld),
                chi2 = chi2,
                sample_size = gwas$n_eff,
                blocks = NULL)
               )

ldsc_h2_est <- ldsc[["h2"]]
ldsc_int_est <- ldsc[['int']]

In [5]:
# Load new data for prediction
pred <- load_bigsnp_from_bed(args$pred)
genotypes <- pred$G

Warning message in load_bigsnp_from_bed(args$pred):
"NAs introduced by coercion"


ERROR: Error: 'infos.chr' should have only positive values.


In [6]:
bed <- args$pred

In [22]:
if (!file.exists.ext(bed, '.bk')) snp_readBed(bed)
basename <- tools::file_path_sans_ext(bed)
rds <- paste0(basename,'.rds')

# if the bk file exists but not the rds, then re-read SNP
if (!file.exists(rds)) {
       write("rds file not available. Retrying snp_readBED", stderr())
       bk <- paste0(basename, ".bk")
       unlink(bk)
       snp_readBed(bed)
}

# attach to the file to current session
big_snp <- snp_attach(rds)

# extract the SNP information from the genotype
autosomes <- paste0('chr',1:22)
map <- big_snp$map[-3]
names(map) <- c("chr", "rsid", "pos", "a1", "a0")

odd_contig <- unique(map$chr[!map$chr %in% autosomes])
odd_contig <- paste0(odd_contig, collapse = '|')
warning(paste0("non standard chromosome contig excluded: ", odd_contig))

# remove non-autosomes (e.g. chr8_KI270821v1_alt)
map <- map[map$chr %in% autosomes,]
big_snp$map <- big_snp$map[big_snp$map$chr %in% autosomes,]

G <- big_snp$genotypes

Warning message in eval(expr, envir, enclos):
"non standard chromosome contig excluded:chr8_KI270821v1_alt"


In [12]:
unique(map$chr)

[1] "chr8"

In [13]:
# Rename the data structures
CHR <- as.numeric(gsub('chr','',map$chr))
POS <- map$pos

# get the CM information from hapmap SNPs
POS2 <- snp_as_genetic_position(CHR, POS, mapdir = "data/prs/1000-genomes-genetic-maps",genetic_map = 'hapmap')

In [ ]:
# ldpred2 inf model
#beta_inf <- snp_ldpred2_inf(snp$corr, gwas, ldsc_h2_est)


In [9]:
#final_pred_inf <- predict_prs(
#       obj = pred,
#       gwas = gwas,
#       effects = beta_inf,
#       ncores = 1,
#       ind_row = 1:100)

In [18]:
head(gwas)
head(pred$map)

,chr,pos,a0,a1,rsid.ss,n_eff,beta_se,p,beta,INFO,MAF,_NUM_ID_.ss,rsid,_NUM_ID_,marker
,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<int>,<chr>,<int>,<chr>
1,chr1,817341,A,G,chr1:817341_A_G,381192,0.011795,0.18683,-0.0155700,NA,0.16040,1,chr1:817341:A:G,1,chr1:817341:A:G
2,chr1,818802,A,G,chr1:818802_A_G,381192,0.012571,0.22785,-0.0151590,NA,0.13027,2,chr1:818802:A:G,2,chr1:818802:A:G
3,chr1,818954,T,C,chr1:818954_T_C,381192,0.012564,0.22805,-0.0151440,NA,0.13057,3,chr1:818954:T:C,3,chr1:818954:T:C
4,chr1,825532,C,T,chr1:825532_C_T,381192,0.011706,0.25218,-0.0134050,NA,0.16099,4,chr1:825532:C:T,4,chr1:825532:C:T
5,chr1,833068,G,A,chr1:833068_G_A,381192,0.013745,0.84022,0.0027711,NA,0.10605,5,chr1:833068:G:A,5,chr1:833068:G:A
7,chr1,843942,A,G,chr1:843942_A_G,381192,0.012648,0.35109,0.0117940,NA,0.12845,7,chr1:843942:A:G,7,chr1:843942:A:G


,chr,rsid,pos,a1,a0
,<chr>,<chr>,<int>,<chr>,<chr>
1,chr22,chr22:15473564:G:A,15473564,A,G
2,chr22,chr22:16390411:A:G,16390411,G,A
3,chr22,chr22:16406147:G:A,16406147,A,G
4,chr22,chr22:16412132:A:G,16412132,G,A
5,chr22,rs5747620,16551808,C,T
6,chr22,chr22:16573137:G:A,16573137,A,G


In [38]:

#res <- match_map(gwas, pred$map, pred$bigsnp)

ori = pred$map
ref = gwas
bigsnp = pred$bigsnp

#match_map <- function(ori, ref, bigsnp){
    
    if (! "marker" %in% colnames(ori)) ori$marker <- get_ldpred_marker(ori)
    if (! "marker" %in% colnames(ref)) ref$marker <- get_ldpred_marker(ref)
    
    # subset ref to things contained in ori
    ref <- ref[ref$marker %in% ori$marker,]
    #ori <- ori[ori$marker %in% ref$marker,]
    
    # match gwas betas with bed map
    indicies <- na.omit(match(ref$marker, ori$marker))
    ori <- ori[indicies,]
    
    nrow(ori); nrow(ref)
    
    # check that all positions match
    stopifnot(all(ori$pos == ref$pos))
    
    
#}


[1] 16185

[1] 1121294

Warning message in ori$pos == ref$pos:
"longer object length is not a multiple of shorter object length"


ERROR: Error: all(ori$pos == ref$pos) is not TRUE


In [ ]:
#if (! "marker" %in% colnames(ori)) ori <- get_ldpred_marker(ori)
#if (! "marker" %in% colnames(ref)) ref <- get_ldpred_marker(ref)

A Sparse Filebacked Big Matrix with 1121294 rows and 1121294 columns.

In [ ]:
# ldpred2 auto model on full gwas
multi_auto <- snp_ldpred2_auto(
   snp$corr,
   gwas,
   h2_init = ldsc_h2_est,
   vec_p_init = 0.5, #seq_log(1e-4, 0.5, 30),
   ncores = 1)



In [ ]:
gwas$effects <- multi_auto
matched <- match_bigsnp_with_gwas(pred, gwas)

sid <- matched$sid
genotypes <- matched$genotype
map <- matched$map

## what is beta_auto??



In [ ]:
# run first estimates
beta_auto <- sapply(multi_auto, function(auto) auto$beta_est)
pred_auto <- big_prodMat(
    genotypes,
    beta_auto,
    ncores = 1)

In [ ]:
# quality controls on chains
sc <- apply(pred_auto, 2, sd)
keep <- abs(sc - median(sc)) < 3 * mad(sc)
final_beta_auto <- rowMeans(beta_auto[, keep])

In [ ]:
final_pred_auto <- big_prodVec(
    genotypes, final_beta_auto,
    ncores = 1)

In [ ]:
# ldpred2 auto model
  #multi_auto <- snp_ldpred2_auto(
  #   snp$corr,
  #   gwas,
  #   h2_init = ldsc_h2_est,
  #   vec_p_init = seq_log(1e-4, 0.5, 30),
  #   ncores = NCORES)

  # run first estimates
  #beta_auto <- sapply(multi_auto, function(auto) auto$beta_est)
  #pred_auto <- big_prodMat(
  #    genotypes,
  #    beta_auto,
  #    ind.col = gwas[["_NUM_ID_"]],
  #    ncores = NCORES)

  # quality controls on chains
  #sc <- apply(pred_auto, 2, sd)
  #keep <- abs(sc - median(sc)) < 3 * mad(sc)
  #final_beta_auto <- rowMeans(beta_auto[, keep])

  # run final estimates

  #final_pred_auto <- big_prodVec(
  #    genotypes, final_beta_auto,
  #    ind.col = gwas[["_NUM_ID_"]],
  #    ncores = NCORES)

  # save parameters
  model <- data.table(
     n_samples = unique(gwas$n),
     n_snps = nrow(gwas),
     ldsc_h2_est = ldsc_h2_est,
     ldsc_int_est = ldsc_int_est,
     inflation = calc_inflation(gwas$P)
     )

  # save results
  PGS <- data.table(
    sid = final_pred_inf$sid,
    prs = final_pred_inf$prs
    )

  write(paste0(args$pred, ".. done! Writing to ", args$out_prefix, ".txt.gz"), stdout())
  fwrite(PGS, file = paste0(args$out_prefix,".txt.gz"), sep = '\t')
  fwrite(model, file = paste0(args$out_prefix,".model"), sep = '\t')

In [ ]:
calc_ld_matrix2 <- function(info_snp, chrs = 1:22, sfbm_file = tempfile(), ld_dir = "data/prs/hapmap/ld/matrix"){

    chrs <- paste0("chr",chrs)
    first_chr <- chrs[1]
    for (chr in chrs) {
        
        # measure time
        start_time <- Sys.time()
        msg = paste0("Retriving LD for chrom ",chr)
        message(msg,"\r",appendLF=FALSE)
        
        # paths to ld-matrix
        path_rds <- file.path(ld_dir, paste0("ld_matrix_",chr,".rds"))
        path_map <- file.path(ld_dir, paste0("ld_matrix_",chr,".txt.gz"))
        
        # retrieve correlation matrix
        corr0 <- readRDS(path_rds)
        map <- fread(path_map)
        map$index <- 1:nrow(map)
        map$marker <- get_ldpred_marker(map)
        
        # subset ld-matrix
        i_index <- na.omit(match(info_snp$marker, map$marker)) 
        matched_map <- map[i_index,]
        j_index <- matched_map$index
        corr0 <- corr0[j_index, j_index]
    
        if (chr == first_chr) {
            ld <- Matrix::colSums(corr0^2)
            corr <- as_SFBM(corr0, sfbm_file)
            outmap <- matched_map
        } else {
            ld <- c(ld, Matrix::colSums(corr0^2))
            corr$add_columns(corr0, nrow(corr))
            outmap <- rbind(outmap, matched_map)
        }
        
        flush.console()
        end_time <- Sys.time()
        elapsed <- round(end_time - start_time,1)
        message(msg," [",elapsed,"s]","\r")
        
    }
    return(list(ld = ld, corr = corr, map = outmap))
}


In [ ]:
match_bigsnp_with_gwas <- function(obj, gwas, ind_row = NULL, ncores = 1){

    # assume that effect index correspond to gwas index
    stopifnot(length(effects) == nrow(gwas))

    # add indexes used for later subsetting
    bed_map <- obj$map
    bed_map$marker <- get_ldpred_marker(bed_map)
    bed_map$index <- 1:nrow(bed_map)
    gwas$marker <- get_ldpred_marker(gwas)
    gwas$index <- 1:nrow(gwas)

    # check that all markers are contained in one another
    stopifnot(sum(gwas$marker %in% bed_map$marker) == sum(bed_map$marker %in% gwas$marker))

    # remove any bed markers not in gwas (assuming
    # that the GWAS has already been qced before)
    bed_map <- bed_map[bed_map$marker %in% gwas$marker,]

    # match gwas betas with bed map
    indicies <- match(bed_map$marker, gwas$marker)
    gwas_matched <- gwas[indicies,]
    gwas_matched <- gwas_matched[!(rowSums(is.na(gwas_matched)) == ncol(gwas_matched)),]

    # check that all positions match
    stopifnot(all(gwas_matched$pos == bed_map$pos))

    # get subset of effects corresponding to bed_map
    matched_effects <- effects[indicies]

    # subset bigsnp to get the right genotypes
    samples <- obj$bigsnp$fam$sample.ID
    if (is.null(ind_row)) ind_row <- 1:length(samples)
    subsetted_bigsnp <- snp_subset(obj$bigsnp, ind.col = bed_map$index, ind.row = ind_row)
    subsetted_bigsnp <- snp_attach(subsetted_bigsnp)
    genotypes <- subsetted_bigsnp$genotypes
    stopifnot(dim(genotypes)[2] == nrow(gwas_matched))

    return(list(G = genotypes, map = gwas_matched, sid =  samples[ind_row]))
}